In [2]:
#!/usr/bin/env python3
"""
Fixes fed_speech.csv rows whose 'content' field contains a raw, unquoted
newline or stray double-quote character — the kind of thing psql's \\copy
rejects with "unquoted newline found in data".

How it works:
  A real record always starts with "<integer>," (the id field). Any line
  that does NOT start that way is treated as a continuation of the
  PREVIOUS record's content — i.e. what should have been a space got
  written as a literal newline instead. Those continuation lines get
  rejoined with a space.

  Once every record is back on one logical line, each is split on the
  first 4 commas (id, date, title, speaker never contain commas, by the
  same convention clean_text() already enforces elsewhere in this
  project) and the remainder is treated as content. Any stray embedded
  double-quote or newline left in content is stripped/collapsed, then
  the field is re-quoted cleanly.

Run this once, inspect the output, then point \\copy at the _fixed file
instead of the original.
"""

import csv
import re
import sys
from pathlib import Path

# Same fix as TOOLS_automate_scrape.py: the csv module's default 128KB
# per-field limit is too small for full speech transcripts stored in the
# 'content' field. Without this, even just re-reading the fixed file for
# the sanity check below fails with "field larger than field limit".
_max_field_size = sys.maxsize
while True:
    try:
        csv.field_size_limit(_max_field_size)
        break
    except OverflowError:
        _max_field_size = int(_max_field_size / 10)

SRC = Path("dataset/fed_speech.csv")
DST = Path("dataset/fed_speech_fixed.csv")

RECORD_START_RE = re.compile(r"^\d+,")


def main():
    raw_lines = SRC.read_text(encoding="utf-8", errors="replace").split("\n")
    if not raw_lines:
        print("File is empty?", file=sys.stderr)
        return

    header = raw_lines[0]
    logical_lines = []
    current = None

    for line in raw_lines[1:]:
        if RECORD_START_RE.match(line):
            if current is not None:
                logical_lines.append(current)
            current = line
        else:
            if current is None:
                continue  # stray blank/garbage line before the first real record
            # This is a continuation of the previous record's content —
            # a newline that should have been a space.
            current += " " + line

    if current is not None:
        logical_lines.append(current)

    raw_data_lines = len(raw_lines) - 1
    print(f"Raw file had {raw_data_lines} physical lines after the header.")
    print(f"Reconstructed into {len(logical_lines)} logical records.")

    fixed_rows = []
    bad = 0
    for i, line in enumerate(logical_lines, start=1):
        parts = line.split(",", 4)
        if len(parts) != 5:
            print(f"  ! Record {i}: expected 5 fields, got {len(parts)} — dropping: {line[:80]!r}...", file=sys.stderr)
            bad += 1
            continue
        id_, date, title, speaker, content = parts
        content = content.strip()
        if content.startswith('"') and content.endswith('"') and len(content) >= 2:
            content = content[1:-1]
        # Same normalization clean_text() applies elsewhere: no embedded
        # quotes or newlines inside a field, whitespace collapsed.
        content = content.replace('"', "").replace("\n", " ").replace("\r", " ")
        content = re.sub(r"\s+", " ", content).strip()
        fixed_rows.append((id_, date, title, speaker, content))

    print(f"{bad} malformed record(s) dropped (logged above).")

    with DST.open("w", newline="", encoding="utf-8") as f:
        f.write(header + "\n")
        for id_, date, title, speaker, content in fixed_rows:
            f.write(f'{id_},{date},{title},{speaker},"{content}"\n')

    print(f"Wrote {len(fixed_rows)} clean record(s) to {DST}")

    # Sanity check: re-parse the file we just wrote with Python's own csv
    # module and confirm every row has exactly 5 columns.
    with DST.open(newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)  # header
        bad_reparse = sum(1 for row in reader if len(row) != 5)
    if bad_reparse:
        print(f"  ! WARNING: {bad_reparse} row(s) in the fixed file still don't parse as 5 columns.", file=sys.stderr)
    else:
        print("Sanity check passed: every row in the fixed file parses as exactly 5 columns.")


if __name__ == "__main__":
    main()

Raw file had 2003 physical lines after the header.
Reconstructed into 2002 logical records.
0 malformed record(s) dropped (logged above).
Wrote 2002 clean record(s) to dataset\fed_speech_fixed.csv
Sanity check passed: every row in the fixed file parses as exactly 5 columns.
